# E-commerce Landing-page A/B Analysis

**Business question:** Did the alternative landing page (`/lander-1`) reduce bounce compared with `/home`, and what evidence is available about purchase conversion?

This is a retrospective analysis of the Maven Fuzzy Factory educational dataset. It combines MySQL extraction and data checks with Python statistical testing. The historical traffic assignment and experiment launch were not conducted as part of this analysis.

## Contents
0. Environment and metric definitions
1. SQL data extraction
2. Session-level data validation
3. Descriptive analysis
4. Sample-ratio check
5. Hypothesis tests and confidence intervals
6. Results and decision interpretation


## Part 0: Environment and metric definitions

| Metric | Definition | Role in this analysis |
| --- | --- | --- |
| Bounce rate | Sessions with exactly one pageview / eligible sessions | Engagement outcome |
| Purchase conversion | Sessions linked to at least one order / eligible sessions | Downstream business outcome |
| Sample-ratio mismatch | Observed session counts versus an assumed 50/50 split | Experiment-quality check |

These definitions organize the historical analysis. They are not claimed to have been preregistered before the original experiment. The tests are two-sided at a 5% significance level, with 95% confidence intervals for B minus A.


In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data"

import numpy as np
import pandas as pd
from scipy.stats import chisquare
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

CSV_PATH = DATA_DIR / "session_level_analysis.csv"
VARIANTS = ["/home", "/lander-1"]
ALPHA = 0.05


## Part 1: SQL data extraction

Run the following SQL in MySQL, then export `session_level_bounce` to `data/session_level_analysis.csv`. The SQL is displayed here as a MySQL step; it is not executed by the Python kernel.

The cohort selects gsearch/nonbrand sessions whose first pageview ID is greater than 23504 and whose landing-page timestamp is before 2012-07-28. The first-pageview and page-count aggregation is constructed before joining orders.

```sql
use mavenfuzzyfactory; 

-- Build a session-level table from pageviews.
create temporary table sessionly_homepage
select website_session_id, 
min(website_pageview_id) landing_page, 
count(website_pageview_id) page_count
from website_pageviews
group by 1;

create temporary table session_level_bounce
select sh.website_session_id, pageview_url landing_page, page_count, 
s.user_id, s.created_at, s.device_type, 
case when page_count = 1 then 1 else 0 end bounced, 
case when order_id is not null then 1 else 0 end purchased
from sessionly_homepage sh 
join website_pageviews p
on sh.landing_page = p.website_pageview_id
join website_sessions s
on p.website_session_id = s.website_session_id
left join orders o 
on s.website_session_id = o.website_session_id
where website_pageview_id > 23504
and p.created_at < '2012-07-28'
and utm_source = 'gsearch' 
and utm_campaign = 'nonbrand';

-- Check session uniqueness and missing key fields.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT website_session_id) AS unique_sessions,
    SUM(user_id IS NULL) AS missing_user_id,
    SUM(landing_page IS NULL) AS missing_landing_page,
    SUM(bounced IS NULL) AS missing_bounced,
    SUM(purchased IS NULL) AS missing_purchased
FROM session_level_bounce;

-- Check repeated users and exposure to both landing-page versions.
-- Repeated users require a dependence-aware analysis.
-- User-cluster resampling must retain the original experiment groups.
-- Cluster bootstrap does not randomly reassign users to treatment/control groups.
-- Alternatively, define a user-level outcome consistently before analysis.
SELECT
    COUNT(*) AS total_users,
    SUM(session_count > 1) AS users_with_multiple_sessions,
    SUM(version_count > 1) AS users_exposed_to_both_versions
FROM (
    SELECT
        user_id,
        COUNT(*) AS session_count,
        COUNT(DISTINCT landing_page) AS version_count
    FROM session_level_bounce
    WHERE user_id IS NOT NULL
    GROUP BY user_id
) u;

-- Inspect device composition by experiment version.
SELECT
    landing_page,
    device_type,
    COUNT(*) AS sessions
FROM session_level_bounce
GROUP BY landing_page, device_type
ORDER BY landing_page, device_type;
```

The current export passes session uniqueness. A session with multiple joined orders could otherwise duplicate rows, so the uniqueness check remains necessary before interpreting session-level rates.


## Part 2: Session-level data validation

Validate the exported row unit, required fields, binary outcomes and user exposure before computing rates. These checks cannot by themselves prove that the original allocation was randomized.


In [2]:
if not CSV_PATH.exists():
    raise FileNotFoundError("Place the authorized session_level_analysis.csv export in the data folder.")
df = pd.read_csv(CSV_PATH)
required_columns = ["website_session_id", "landing_page", "page_count", "user_id", "created_at", "device_type", "bounced", "purchased"]
assert set(required_columns).issubset(df.columns), "Required columns are missing."
assert not df[required_columns].isna().any().any(), "Required fields contain missing values."
assert df["website_session_id"].is_unique, "The export must contain one row per session."
assert set(df["landing_page"].unique()) == set(VARIANTS), "Unexpected landing-page cohort."
assert df["bounced"].isin([0, 1]).all() and df["purchased"].isin([0, 1]).all()
assert df["bounced"].eq(df["page_count"].eq(1).astype(int)).all(), "Bounce flags disagree with page counts."
pd.DataFrame({"check": ["rows", "unique_sessions", "unique_users"], "value": [len(df), df.website_session_id.nunique(), df.user_id.nunique()]})


             check  value
0             rows   4576
1  unique_sessions   4576
2     unique_users   4576

In [3]:
user_checks = df.groupby("user_id").agg(sessions=("website_session_id", "size"), versions=("landing_page", "nunique"))
exposure_checks = pd.Series({
    "users_with_multiple_sessions": int(user_checks.sessions.gt(1).sum()),
    "users_exposed_to_both_versions": int(user_checks.versions.gt(1).sum()),
}, name="count")
print(exposure_checks.to_string())
if user_checks.sessions.gt(1).any():
    raise ValueError("Repeated users require an appropriate user-level or clustered analysis before using the independent-proportion tests below.")


users_with_multiple_sessions      0
users_exposed_to_both_versions    0


## Part 3: Descriptive analysis

Inspect device composition and summarize each metric by landing-page version. A numerical difference is not, by itself, evidence of an effect.


In [4]:
device_summary = df.groupby(["landing_page", "device_type"]).size().unstack(fill_value=0).reindex(VARIANTS)
device_summary


device_type   desktop  mobile
landing_page                 
/home            1721     540
/lander-1        1777     538

In [5]:
summary = df.groupby("landing_page").agg(
    sessions=("website_session_id", "size"),
    bounced_sessions=("bounced", "sum"),
    purchased_sessions=("purchased", "sum"),
).reindex(VARIANTS)
summary["bounce_rate"] = summary.bounced_sessions / summary.sessions
summary["purchase_rate"] = summary.purchased_sessions / summary.sessions
summary


              sessions  bounced_sessions  purchased_sessions  bounce_rate  \
landing_page                                                                
/home             2261              1319                  72     0.583370   
/lander-1         2315              1232                  94     0.532181   

              purchase_rate  
landing_page                 
/home              0.031844  
/lander-1          0.040605  

## Part 4: Sample-ratio check

Use a chi-square goodness-of-fit test against the assumed 50/50 allocation. A non-significant result means this check does not detect a mismatch; it does not establish that all aspects of the experiment are valid.


In [6]:
total = int(summary["sessions"].sum())
srm = chisquare(f_obs=summary["sessions"].to_numpy(), f_exp=[total / 2, total / 2])
print(f"SRM p-value: {srm.pvalue:.4f}")


SRM p-value: 0.4247


## Part 5: Hypothesis tests and confidence intervals

For each outcome, the null hypothesis is that the two population proportions are equal. Use a two-proportion z-test and a Newcombe interval for the B-minus-A difference.


In [7]:
n_a = int(summary.loc["/home", "sessions"])
n_b = int(summary.loc["/lander-1", "sessions"])
bounce_a = int(summary.loc["/home", "bounced_sessions"])
bounce_b = int(summary.loc["/lander-1", "bounced_sessions"])
purchase_a = int(summary.loc["/home", "purchased_sessions"])
purchase_b = int(summary.loc["/lander-1", "purchased_sessions"])


In [8]:
def compare_proportions(label, count_a, count_b, n_a, n_b):
    rate_a, rate_b = count_a / n_a, count_b / n_b
    z_stat, p_value = proportions_ztest(count=[count_b, count_a], nobs=[n_b, n_a], alternative="two-sided")
    ci_low, ci_high = confint_proportions_2indep(
        count1=count_b, nobs1=n_b, count2=count_a, nobs2=n_a,
        compare="diff", method="newcomb", alpha=ALPHA,
    )
    return {"metric": label, "rate_a": rate_a, "rate_b": rate_b,
            "difference_pp": (rate_b-rate_a)*100,
            "ci_low_pp": ci_low*100, "ci_high_pp": ci_high*100,
            "z_stat": float(z_stat), "p_value": float(p_value)}

results = pd.DataFrame([
    compare_proportions("Bounce rate", bounce_a, bounce_b, n_a, n_b),
    compare_proportions("Purchase conversion", purchase_a, purchase_b, n_a, n_b),
]).set_index("metric")
results


                       rate_a    rate_b  difference_pp  ci_low_pp  ci_high_pp  \
metric                                                                          
Bounce rate          0.583370  0.532181      -5.118876  -7.984453   -2.240845   
Purchase conversion  0.031844  0.040605       0.876043  -0.212616    1.971381   

                       z_stat   p_value  
metric                                   
Bounce rate         -3.485585  0.000491  
Purchase conversion  1.584605  0.113056  

## Part 6: Results and decision interpretation

The historical export contains 4,576 unique sessions/users. The alternative page has a lower observed bounce rate: 53.22% versus 58.34%, a difference of -5.12 percentage points (95% CI approximately [-7.98, -2.24]; p approximately 0.000491).

Purchase conversion is 4.06% versus 3.18%, a +0.88 percentage-point difference. Its interval includes zero (approximately [-0.21, 1.97] percentage points; p approximately 0.113056), so the analysis does not establish a purchase-conversion increase at the 5% level.

The bounce result supports improved engagement in this historical comparison. It does not establish the mechanism behind the change, prove a downstream purchase benefit, or establish equivalence/no harm for purchases. A rollout decision would also require the original assignment information and business decision criteria. No additional live test or sample-size plan is claimed here.
